# Export selected CardioSentry multi-label ECG images to S3

This EC2 notebook creates a **copy-only** derived dataset from the already-extracted CardioSentry master dataset. It selects exactly these binary-label combinations from `train`, `val`, and `test`:

- AF + LVH
- STEMI + LVH
- STEMI + AF
- STEMI + AF + LVH

The notebook never deletes, moves, renames, or writes into the master dataset. Copied images are placed under `images/<split>/<combination>/`, labels are saved in CSV manifests, SHA-256 hashes are recorded, and the complete derived dataset is uploaded to a versioned S3 prefix. It supports the EC2 layouts used by both `convnext_tiny_master_v1` and `convnextv2_tiny_master_v1`.

Run all cells from top to bottom. The EC2 IAM role needs `s3:PutObject`, `s3:GetObject`, and `s3:ListBucket` for the destination prefix. If auto-discovery does not find your layout, set `CS_INDEX_CSV` and `CS_IMAGES_ROOT` before starting the kernel.


In [ ]:
# 1. Imports and configuration
import concurrent.futures as futures
import hashlib
import json
import mimetypes
import os
import shutil
import socket
import threading
from datetime import datetime, timezone
from pathlib import Path

try:
    import boto3
    import pandas as pd
except ModuleNotFoundError as exc:
    raise ModuleNotFoundError(
        f"Missing {exc.name}. Install once with: %pip install boto3 pandas"
    ) from exc

def env_flag(name, default="1"):
    return os.environ.get(name, default).strip().lower() not in {"0", "false", "no", "off", ""}

def sha256_file(path, chunk_bytes=8 * 1024 * 1024):
    digest = hashlib.sha256()
    with Path(path).open("rb") as handle:
        for chunk in iter(lambda: handle.read(chunk_bytes), b""):
            digest.update(chunk)
    return digest.hexdigest()

PROJECT_ROOT = Path(os.environ.get("CARDIOSENTRY_PROJECT_ROOT", str(Path.home() / "CardioSentry"))).expanduser().resolve()
PREFERRED_STORAGE_ROOT = Path("/mnt/cardiosentry")
if PREFERRED_STORAGE_ROOT.is_dir() and os.access(PREFERRED_STORAGE_ROOT, os.W_OK):
    DEFAULT_OUTPUT_PARENT = PREFERRED_STORAGE_ROOT / "derived-datasets"
else:
    DEFAULT_OUTPUT_PARENT = PROJECT_ROOT / "derived-datasets"

DATASET_NAME = os.environ.get("CS_SUBSET_NAME", "af_lvh_stemi_combinations_v1").strip()
if not DATASET_NAME or any(part in DATASET_NAME for part in ("/", "\\")):
    raise ValueError("CS_SUBSET_NAME must be one non-empty path component")
if "_CS_EXPORT_RUN_ID" not in globals():
    _CS_EXPORT_RUN_ID = os.environ.get(
        "CS_SUBSET_RUN_ID", datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    ).strip()
RUN_ID = _CS_EXPORT_RUN_ID
if not RUN_ID or any(part in RUN_ID for part in ("/", "\\")):
    raise ValueError("CS_SUBSET_RUN_ID must be one non-empty path component")

OUTPUT_PARENT = Path(os.environ.get("CS_SUBSET_OUTPUT_ROOT", str(DEFAULT_OUTPUT_PARENT))).expanduser().resolve()
DATASET_DIR = (OUTPUT_PARENT / DATASET_NAME / RUN_ID).resolve()
COPY_WORKERS = max(1, int(os.environ.get("CS_COPY_WORKERS", "8")))
UPLOAD_WORKERS = max(1, int(os.environ.get("CS_UPLOAD_WORKERS", "16")))
STRICT_MASTER_AUDIT = env_flag("CS_STRICT_MASTER_AUDIT", "1")

S3_BUCKET = os.environ.get("CS_S3_BUCKET", "cardiosentry-fcmae-758821198776-us-east-1-an").strip()
S3_BASE_PREFIX = os.environ.get(
    "CS_S3_SUBSET_PREFIX", "CardioSentry/datasets/af_lvh_stemi_combinations_v1"
).strip("/")
S3_RUN_PREFIX = f"{S3_BASE_PREFIX}/{RUN_ID}"
AWS_REGION = os.environ.get("AWS_REGION", os.environ.get("AWS_DEFAULT_REGION", "us-east-1"))

if not S3_BUCKET or not S3_BASE_PREFIX:
    raise ValueError("CS_S3_BUCKET and CS_S3_SUBSET_PREFIX must be non-empty")

print("Derived dataset directory:", DATASET_DIR)
print("S3 destination:           ", f"s3://{S3_BUCKET}/{S3_RUN_PREFIX}/")
print("Strict master audit:     ", STRICT_MASTER_AUDIT)


In [ ]:
# 2. Locate the reconstructed index and its extracted images
explicit_index = os.environ.get("CS_INDEX_CSV") or os.environ.get("CS_RECONSTRUCTED_INDEX_CSV")
index_candidates = []
if explicit_index:
    index_candidates.append(Path(explicit_index).expanduser().resolve())
index_candidates.extend([
    Path("/mnt/cardiosentry/master-dataset/extracted/reconstructed_index.csv"),
    Path("/mnt/cardiosentry/master_dataset/extracted/reconstructed_index.csv"),
    PROJECT_ROOT / "master-dataset/extracted/reconstructed_index.csv",
    PROJECT_ROOT / "master_dataset/extracted/reconstructed_index.csv",
    PROJECT_ROOT / "experiments/convnextv2_tiny_master_v1/data/reconstructed_index.csv",
    PROJECT_ROOT / "experiments/convnext_tiny_master_v1/data/reconstructed_index.csv",
    PROJECT_ROOT / "datasets/synthetic_images_v1/index.csv",
])

# Preserve order while removing duplicate paths. An explicit path is authoritative.
index_candidates = list(dict.fromkeys(path.resolve() for path in index_candidates))
if explicit_index and not index_candidates[0].is_file():
    raise FileNotFoundError(f"Explicit CS_INDEX_CSV does not exist: {index_candidates[0]}")
INDEX_CSV = next((path for path in index_candidates if path.is_file()), None)
if INDEX_CSV is None:
    searched = "\n".join(f"  - {path}" for path in index_candidates)
    raise FileNotFoundError(
        "Could not find reconstructed_index.csv (or index.csv). Set CS_INDEX_CSV. Searched:\n" + searched
    )

index_df = pd.read_csv(INDEX_CSV, low_memory=False)
required_columns = {
    "image_path", "record", "record_id", "original_id", "cls",
    "STEMI", "AF", "LVH", "NORMAL", "split", "source",
    "render_k", "patient_id",
}
missing_columns = sorted(required_columns - set(index_df.columns))
if missing_columns:
    raise ValueError(f"Index is missing required columns: {missing_columns}")

def indexed_basename(row):
    raw = Path(str(row["image_path"])).name
    if Path(raw).suffix.lower() in {".jpg", ".jpeg", ".png"}:
        return raw
    return f"{row['record']}_a{int(row['render_k'])}.jpg"

explicit_images = os.environ.get("CS_IMAGES_ROOT")
images_candidates = []
if explicit_images:
    images_candidates.append(Path(explicit_images).expanduser().resolve())
images_candidates.extend([
    INDEX_CSV.parent / "images",
    Path("/mnt/cardiosentry/master-dataset/extracted/images"),
    Path("/mnt/cardiosentry/master_dataset/extracted/images"),
    PROJECT_ROOT / "master-dataset/extracted/images",
    PROJECT_ROOT / "master_dataset/extracted/images",
    PROJECT_ROOT / "experiments/convnextv2_tiny_master_v1/data/images",
    PROJECT_ROOT / "experiments/convnext_tiny_master_v1/data/images",
])
images_candidates = list(dict.fromkeys(path.resolve() for path in images_candidates))

sample = pd.concat(
    [part.head(32) for _, part in index_df.groupby(index_df["split"].astype(str).str.lower())],
    ignore_index=True,
)
def candidate_match_count(root):
    return sum(
        (root / str(row["split"]).lower().strip() / indexed_basename(row)).is_file()
        for _, row in sample.iterrows()
    )

if explicit_images:
    IMAGES_ROOT = images_candidates[0]
    if not IMAGES_ROOT.is_dir():
        raise FileNotFoundError(f"Explicit CS_IMAGES_ROOT is not a directory: {IMAGES_ROOT}")
else:
    scored = [(candidate_match_count(root), root) for root in images_candidates if root.is_dir()]
    IMAGES_ROOT = max(scored, default=(0, None), key=lambda item: item[0])[1]

if IMAGES_ROOT is None or candidate_match_count(IMAGES_ROOT) != len(sample):
    scores = {str(root): candidate_match_count(root) for root in images_candidates if root.is_dir()}
    raise FileNotFoundError(
        "Could not match the index to an extracted images root. Set CS_IMAGES_ROOT. "
        f"Candidate sample matches: {scores}"
    )

print("Index:      ", INDEX_CSV)
print("Index rows: ", f"{len(index_df):,}")
print("Images root:", IMAGES_ROOT)


In [ ]:
# 3. Validate labels and build the exact copy plan
LABELS = ["NORMAL", "STEMI", "AF", "LVH"]
PATHOLOGY_LABELS = ["STEMI", "AF", "LVH"]
TARGET_BY_BITS = {
    (0, 1, 1): ("AF + LVH", "af_lvh"),
    (1, 0, 1): ("STEMI + LVH", "stemi_lvh"),
    (1, 1, 0): ("STEMI + AF", "stemi_af"),
    (1, 1, 1): ("STEMI + AF + LVH", "stemi_af_lvh"),
}
EXPECTED_SELECTED_IMAGES = {
    "AF + LVH": 8_406,
    "STEMI + LVH": 1_394,
    "STEMI + AF": 587,
    "STEMI + AF + LVH": 157,
}
EXPECTED_MASTER_IMAGES_BY_SPLIT = {"train": 121_844, "val": 3_434, "test": 3_330}

index_df["split"] = index_df["split"].astype(str).str.lower().str.strip()
if set(index_df["split"].unique()) != {"train", "val", "test"}:
    raise ValueError(f"Expected train/val/test only; got {sorted(index_df['split'].unique())}")
for label in LABELS:
    index_df[label] = pd.to_numeric(index_df[label], errors="raise").astype("int8")
    if not index_df[label].isin([0, 1]).all():
        raise ValueError(f"{label} contains a value outside 0/1")

pathology_count = index_df[PATHOLOGY_LABELS].sum(axis=1)
normal_is_valid = index_df["NORMAL"].eq(1) == pathology_count.eq(0)
if not normal_is_valid.all():
    raise ValueError("The index violates NORMAL = no pathology semantics")

for identity in ("record", "record_id", "patient_id"):
    split_count = index_df.groupby(identity, dropna=False)["split"].nunique()
    if split_count.gt(1).any():
        raise ValueError(f"{identity} crosses dataset splits: {split_count[split_count.gt(1)].index[:5].tolist()}")
for field in ["split", *LABELS]:
    changes = index_df.groupby("record", dropna=False)[field].nunique(dropna=False)
    if changes.gt(1).any():
        raise ValueError(f"{field} changes across render variants of a record")

if STRICT_MASTER_AUDIT:
    observed_split_counts = {str(k): int(v) for k, v in index_df.groupby("split").size().items()}
    if observed_split_counts != EXPECTED_MASTER_IMAGES_BY_SPLIT:
        raise RuntimeError(
            f"This is not the expected v1 master index. Expected {EXPECTED_MASTER_IMAGES_BY_SPLIT}, "
            f"observed {observed_split_counts}. Set CS_STRICT_MASTER_AUDIT=0 only intentionally."
        )

label_bits = list(zip(index_df["STEMI"], index_df["AF"], index_df["LVH"]))
mapped = [TARGET_BY_BITS.get(tuple(int(value) for value in bits)) for bits in label_bits]
selected_mask = pd.Series([value is not None for value in mapped], index=index_df.index)
selected = index_df.loc[selected_mask].copy()
selected["combination"] = [value[0] for value in mapped if value is not None]
selected["combination_slug"] = [value[1] for value in mapped if value is not None]

observed_combinations = {str(k): int(v) for k, v in selected.groupby("combination").size().items()}
if STRICT_MASTER_AUDIT and observed_combinations != EXPECTED_SELECTED_IMAGES:
    raise RuntimeError(
        f"Selected combination counts differ from the verified master corpus: {observed_combinations}"
    )

selected["_source_path"] = selected.apply(
    lambda row: str((IMAGES_ROOT / row["split"] / indexed_basename(row)).resolve()), axis=1
)
selected["image_path"] = selected.apply(
    lambda row: (Path("images") / row["split"] / row["combination_slug"] / indexed_basename(row)).as_posix(),
    axis=1,
)
if selected["image_path"].duplicated().any():
    duplicate = selected.loc[selected["image_path"].duplicated(False), "image_path"].head().tolist()
    raise ValueError(f"Destination path collision: {duplicate}")

missing_sources = [path for path in selected["_source_path"] if not Path(path).is_file()]
if missing_sources:
    raise FileNotFoundError(f"{len(missing_sources):,} selected images are missing; first: {missing_sources[:5]}")
if DATASET_DIR == IMAGES_ROOT or IMAGES_ROOT in DATASET_DIR.parents or DATASET_DIR in IMAGES_ROOT.parents:
    raise ValueError("Derived DATASET_DIR and master images root must not contain one another")

selected["_source_bytes"] = selected["_source_path"].map(lambda value: Path(value).stat().st_size)
selected["_source_mtime_ns"] = selected["_source_path"].map(lambda value: Path(value).stat().st_mtime_ns)
selected = selected.sort_values(
    ["split", "combination_slug", "record", "render_k"], kind="stable"
).reset_index(drop=True)

plan_summary = selected.groupby(["split", "combination"], sort=False).agg(
    images=("record", "size"), records=("record", "nunique")
).astype(int)
print(plan_summary.to_string())
print(f"\nReady to copy {len(selected):,} images ({selected['_source_bytes'].sum() / 2**30:.2f} GiB).")


In [ ]:
# 4. Copy image bytes into the derived dataset (resumable; source is opened read-only)
DATASET_DIR.mkdir(parents=True, exist_ok=True)
missing_copy_bytes = sum(
    int(source_bytes)
    for source_bytes, relative_path in selected[["_source_bytes", "image_path"]].itertuples(index=False, name=None)
    if not (DATASET_DIR / relative_path).exists()
)
required_free_bytes = int(missing_copy_bytes * 1.05 + 256 * 2**20)
available_free_bytes = shutil.disk_usage(DATASET_DIR).free
if available_free_bytes < required_free_bytes:
    raise RuntimeError(
        f"Insufficient derived-dataset space: {available_free_bytes / 2**30:.2f} GiB free, "
        f"{required_free_bytes / 2**30:.2f} GiB required for remaining copies and margin"
    )
for relative_dir in selected[["split", "combination_slug"]].drop_duplicates().itertuples(index=False):
    (DATASET_DIR / "images" / relative_dir.split / relative_dir.combination_slug).mkdir(parents=True, exist_ok=True)

def copy_and_hash(item):
    row_number, source_text, relative_text, expected_bytes = item
    source = Path(source_text)
    destination = DATASET_DIR / relative_text
    expected_bytes = int(expected_bytes)

    if destination.exists():
        if not destination.is_file() or destination.stat().st_size != expected_bytes:
            raise RuntimeError(f"Existing derived file conflicts with source: {destination}")
        source_digest = sha256_file(source)
        destination_digest = sha256_file(destination)
        if destination_digest != source_digest:
            raise RuntimeError(f"Existing derived file checksum conflicts with source: {destination}")
        return row_number, destination_digest, expected_bytes, "verified_existing"

    temporary = destination.with_name(
        f".{destination.name}.part.{os.getpid()}.{threading.get_ident()}"
    )
    digest = hashlib.sha256()
    copied_bytes = 0
    try:
        with source.open("rb") as reader, temporary.open("xb") as writer:
            for chunk in iter(lambda: reader.read(8 * 1024 * 1024), b""):
                writer.write(chunk)
                digest.update(chunk)
                copied_bytes += len(chunk)
            writer.flush()
            os.fsync(writer.fileno())
        if copied_bytes != expected_bytes:
            raise RuntimeError(f"Source size changed while copying: {source}")
        os.replace(temporary, destination)
        shutil.copystat(source, destination, follow_symlinks=True)
        return row_number, digest.hexdigest(), copied_bytes, "copied"
    finally:
        if temporary.exists():
            temporary.unlink()

copy_items = [
    (idx, row["_source_path"], row["image_path"], row["_source_bytes"])
    for idx, row in selected.iterrows()
]
copy_results = {}
with futures.ThreadPoolExecutor(max_workers=COPY_WORKERS) as executor:
    for completed, result in enumerate(executor.map(copy_and_hash, copy_items), start=1):
        row_number, digest, byte_count, status = result
        copy_results[row_number] = {"sha256": digest, "bytes": byte_count, "status": status}
        if completed % 1000 == 0 or completed == len(copy_items):
            print(f"Copied/verified {completed:,}/{len(copy_items):,}", flush=True)

selected["sha256"] = [copy_results[idx]["sha256"] for idx in selected.index]
selected["bytes"] = [copy_results[idx]["bytes"] for idx in selected.index]

# Confirm that copying did not alter source sizes or modification timestamps.
changed_sources = []
for source_text, expected_bytes, expected_mtime_ns in selected[
    ["_source_path", "_source_bytes", "_source_mtime_ns"]
].itertuples(index=False, name=None):
    source_stat = Path(source_text).stat()
    if source_stat.st_size != expected_bytes or source_stat.st_mtime_ns != expected_mtime_ns:
        changed_sources.append(source_text)
if changed_sources:
    raise RuntimeError(f"A master source changed during the copy: {changed_sources[:5]}")

print(f"Copy complete: {len(selected):,} images, {selected['bytes'].sum() / 2**30:.2f} GiB")


In [ ]:
# 5. Write labels, split manifests, provenance, and local completion metadata
def atomic_write_text(path, text):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(text, encoding="utf-8")
    os.replace(temporary, path)

manifest = selected.copy()
manifest.insert(0, "dataset_run_id", RUN_ID)
manifest.insert(1, "s3_uri", manifest["image_path"].map(lambda value: f"s3://{S3_BUCKET}/{S3_RUN_PREFIX}/{value}"))
manifest.insert(2, "master_image_basename", manifest["_source_path"].map(lambda value: Path(value).name))
# Human-readable ECG/augmentation identity. `render_k` remains the original
# zero-based value used by filenames (_a0 ... _a3); augmentation_number is one-based.
manifest.insert(3, "ecg_record", manifest["record"].astype(str))
manifest.insert(4, "augmentation_number", pd.to_numeric(manifest["render_k"], errors="raise").astype(int) + 1)
manifest.insert(5, "render_variant", "a" + pd.to_numeric(manifest["render_k"], errors="raise").astype(int).astype(str))
manifest.insert(6, "ecg_augmentation_id", manifest["ecg_record"] + "_augmentation_" + manifest["augmentation_number"].astype(str))
internal_columns = [column for column in manifest.columns if column.startswith("_")]
manifest = manifest.drop(columns=internal_columns)

labels_path = DATASET_DIR / "labels.csv"
temporary_labels = labels_path.with_suffix(".csv.tmp")
manifest.to_csv(temporary_labels, index=False)
os.replace(temporary_labels, labels_path)
for split in ("train", "val", "test"):
    split_path = DATASET_DIR / f"labels_{split}.csv"
    temporary_split = split_path.with_suffix(".csv.tmp")
    manifest.loc[manifest["split"].eq(split)].to_csv(temporary_split, index=False)
    os.replace(temporary_split, split_path)

index_stat_after = INDEX_CSV.stat()
summary = {
    "dataset_name": DATASET_NAME,
    "run_id": RUN_ID,
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "created_on_host": socket.gethostname(),
    "copy_only": True,
    "master_dataset_writes_or_deletes": 0,
    "source_index_csv": str(INDEX_CSV),
    "source_index_sha256": sha256_file(INDEX_CSV),
    "source_images_root": str(IMAGES_ROOT),
    "labels": LABELS,
    "selection_rule": "exact binary combinations of STEMI, AF, and LVH; NORMAL must be 0",
    "selected_combinations": list(TARGET_BY_BITS.values()),
    "image_count": int(len(manifest)),
    "unique_record_count": int(manifest["record"].nunique()),
    "total_image_bytes": int(manifest["bytes"].sum()),
    "images_by_split_and_combination": {
        split: {str(key): int(value) for key, value in part.groupby("combination").size().items()}
        for split, part in manifest.groupby("split")
    },
    "records_by_split_and_combination": {
        split: {str(key): int(value) for key, value in part.groupby("combination")["record"].nunique().items()}
        for split, part in manifest.groupby("split")
    },
    "s3_dataset_uri": f"s3://{S3_BUCKET}/{S3_RUN_PREFIX}/",
}
atomic_write_text(DATASET_DIR / "dataset_summary.json", json.dumps(summary, indent=2, sort_keys=True) + "\n")

readme = f"""# CardioSentry selected multi-label ECG dataset

This is a copy-only derivative of the v1 master image dataset.

Included exact label combinations: AF + LVH, STEMI + LVH, STEMI + AF, and STEMI + AF + LVH.
Splits are preserved from the master index: train, val, and test.
The `labels.csv` file is authoritative. Per-split copies are `labels_train.csv`, `labels_val.csv`, and `labels_test.csv`.
Images are stored at `images/<split>/<combination_slug>/<filename>`.
Every manifest row includes `ecg_record`, one-based `augmentation_number`, `render_variant`, and `ecg_augmentation_id`. The original zero-based `render_k` is retained because it matches filename suffixes such as `_a0`.
Every manifest row also includes the independent binary columns NORMAL, STEMI, AF, and LVH plus a SHA-256 digest.

Dataset run: {RUN_ID}
Source index SHA-256: {summary['source_index_sha256']}
S3 destination: {summary['s3_dataset_uri']}
"""
atomic_write_text(DATASET_DIR / "README.md", readme)

# A line-oriented digest file makes independent verification easy after download.
digest_lines = [f"{row.sha256}  {row.image_path}" for row in manifest.itertuples(index=False)]
atomic_write_text(DATASET_DIR / "IMAGE_SHA256SUMS", "\n".join(digest_lines) + "\n")

print("Local derived dataset is complete:", DATASET_DIR)
print(json.dumps(summary["images_by_split_and_combination"], indent=2))


In [ ]:
# 6. Upload every derived file to S3 and verify exact key/size coverage
s3_client = boto3.client("s3", region_name=AWS_REGION)

payload_before_upload_manifest = sorted(
    path for path in DATASET_DIR.rglob("*")
    if path.is_file() and ".part." not in path.name and not path.name.endswith(".tmp")
)
image_hashes = dict(zip(manifest["image_path"], manifest["sha256"]))
upload_records = []
for path in payload_before_upload_manifest:
    relative = path.relative_to(DATASET_DIR).as_posix()
    upload_records.append({
        "relative_path": relative,
        "s3_key": f"{S3_RUN_PREFIX}/{relative}",
        "bytes": int(path.stat().st_size),
        "sha256": image_hashes.get(relative) or sha256_file(path),
    })
upload_manifest_payload = {
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "bucket": S3_BUCKET,
    "prefix": S3_RUN_PREFIX,
    "object_count_excluding_control_files": len(upload_records),
    "objects": upload_records,
}
upload_manifest_path = DATASET_DIR / "s3_upload_manifest.json"
atomic_write_text(upload_manifest_path, json.dumps(upload_manifest_payload, indent=2, sort_keys=True) + "\n")
completion_payload = {
    "status": "complete",
    "created_at_utc": datetime.now(timezone.utc).isoformat(),
    "dataset_uri": f"s3://{S3_BUCKET}/{S3_RUN_PREFIX}/",
    "image_count": int(len(manifest)),
    "upload_manifest_sha256": sha256_file(upload_manifest_path),
}
completion_path = DATASET_DIR / "UPLOAD_COMPLETE.json"
atomic_write_text(completion_path, json.dumps(completion_payload, indent=2, sort_keys=True) + "\n")

payload = sorted(
    path for path in DATASET_DIR.rglob("*")
    if path.is_file() and path != completion_path
    and ".part." not in path.name and not path.name.endswith(".tmp")
)

def upload_one(path):
    relative = path.relative_to(DATASET_DIR).as_posix()
    key = f"{S3_RUN_PREFIX}/{relative}"
    content_type = mimetypes.guess_type(path.name)[0] or "application/octet-stream"
    s3_client.upload_file(
        str(path), S3_BUCKET, key, ExtraArgs={"ContentType": content_type}
    )
    return key, path.stat().st_size

uploaded = {}
try:
    with futures.ThreadPoolExecutor(max_workers=UPLOAD_WORKERS) as executor:
        for completed, result in enumerate(executor.map(upload_one, payload), start=1):
            key, byte_count = result
            uploaded[key] = int(byte_count)
            if completed % 1000 == 0 or completed == len(payload):
                print(f"Uploaded {completed:,}/{len(payload):,}", flush=True)
except Exception as exc:
    raise RuntimeError(
        f"S3 upload failed. Check the EC2 IAM role and destination s3://{S3_BUCKET}/{S3_RUN_PREFIX}/"
    ) from exc

expected_payload_remote = {
    f"{S3_RUN_PREFIX}/{path.relative_to(DATASET_DIR).as_posix()}": int(path.stat().st_size)
    for path in payload
}
observed_remote = {}
paginator = s3_client.get_paginator("list_objects_v2")
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_RUN_PREFIX + "/"):
    for obj in page.get("Contents", []):
        observed_remote[obj["Key"]] = int(obj["Size"])
completion_key = f"{S3_RUN_PREFIX}/UPLOAD_COMPLETE.json"
observed_payload_remote = {key: value for key, value in observed_remote.items() if key != completion_key}
if observed_payload_remote != expected_payload_remote:
    missing = sorted(set(expected_payload_remote) - set(observed_payload_remote))[:10]
    unexpected = sorted(set(observed_payload_remote) - set(expected_payload_remote))[:10]
    wrong_size = [key for key in expected_payload_remote.keys() & observed_payload_remote.keys() if expected_payload_remote[key] != observed_payload_remote[key]][:10]
    raise RuntimeError(
        f"S3 verification failed. Missing={missing}; unexpected={unexpected}; wrong_size={wrong_size}"
    )

# The completion marker is deliberately uploaded only after every payload key is verified.
upload_one(completion_path)
remote_completion = s3_client.head_object(Bucket=S3_BUCKET, Key=completion_key)
if int(remote_completion["ContentLength"]) != completion_path.stat().st_size:
    raise RuntimeError("Remote completion marker size verification failed")

expected_remote = {**expected_payload_remote, completion_key: int(completion_path.stat().st_size)}
observed_remote = {}
for page in paginator.paginate(Bucket=S3_BUCKET, Prefix=S3_RUN_PREFIX + "/"):
    for obj in page.get("Contents", []):
        observed_remote[obj["Key"]] = int(obj["Size"])
if observed_remote != expected_remote:
    raise RuntimeError("Final S3 object-set verification failed after completion marker upload")

print("S3 upload and exact key/size verification complete.")
print(f"Dataset URI: s3://{S3_BUCKET}/{S3_RUN_PREFIX}/")
print(f"Objects:     {len(expected_remote):,}")
print(f"Images:      {len(manifest):,}")


## Result

When the final cell prints `S3 upload and exact key/size verification complete`, the derived copy is ready. `UPLOAD_COMPLETE.json` is written last in the local build and included in the verified S3 object set; consumers can use it as the completion marker. The source master dataset remains in place and unchanged.
